In [1]:
from pyspark.sql import SparkSession

# 1. Initialize Spark 
# (All Iceberg, S3, and JDBC configs are automatically inherited from spark-defaults.conf!)
spark = SparkSession.builder.appName("Iceberg-Lakehouse-Demo").getOrCreate()

# Silence verbose Spark/Iceberg warnings
spark.sparkContext.setLogLevel("ERROR")


26/10/10 11:20:55 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Each Query is executed in SEQUENCE

SEQ 1:
CREATE NAMESPACE IF NOT EXISTS local.urbanfleet;
USE local.urbanfleet;


CREATE TABLE local.urbanfleet.trips (
    trip_id           BIGINT,
    driver_id         BIGINT,
    rider_id          BIGINT,
    pickup_zip        STRING,
    dropoff_zip       STRING,
    fare_amount       DECIMAL(10, 2),
    trip_status       STRING, -- ONGOING, COMPLETED, CANCELLED
    trip_timestamp    TIMESTAMP,
    last_updated      TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(trip_timestamp))
TBLPROPERTIES (
    'format-version' = '2',
    'write.update.mode' = 'merge-on-read',
    'write.delete.mode' = 'merge-on-read',
    'write.merge.mode'  = 'merge-on-read'
);


SEQ 2:
INSERT INTO local.urbanfleet.trips VALUES
(101, 5001, 9001, '10001', '10002', 25.50, 'COMPLETED', TIMESTAMP '2025-01-10 08:00:00', TIMESTAMP '2025-01-10 08:30:00'),
(102, 5002, 9002, '10003', '10004', 15.00, 'COMPLETED', TIMESTAMP '2025-01-10 09:15:00', TIMESTAMP '2025-01-10 09:40:00'),
(103, 5001, 9003, '10001', '10005', 32.75, 'ONGOING',   TIMESTAMP '2025-01-11 14:00:00', TIMESTAMP '2025-01-11 14:00:00'),
(104, 5003, 9004, '10006', '10007', 8.20,  'CANCELLED', TIMESTAMP '2025-01-11 16:20:00', TIMESTAMP '2025-01-11 16:22:00'),
(105, 5004, 9005, '10008', '10009', 45.00, 'COMPLETED', TIMESTAMP '2025-01-12 22:00:00', TIMESTAMP '2025-01-12 22:45:00');

SEQ 3:
UPDATE local.urbanfleet.trips
SET trip_status = 'COMPLETED', last_updated = current_timestamp()
WHERE trip_id = 103;

SEQ 4:
DELETE FROM local.urbanfleet.trips
WHERE trip_id = 104;


In [28]:
# Define the full S3A path to your MinIO file

## MANIFEST LIST
manifest_list_file_path = "s3a://warehouse/local/urbanfleet/trips/metadata/snap-3529268444000415922-1-afb2b273-41da-4581-8227-9c8ace235b81.avro"

# Read the Avro file into a DataFrame
df = spark.read.format("avro").load(manifest_list_file_path)
# [span_1](start_span)[span_1](end_span)

# Print the schema (this displays all column names and data types)
df.printSchema()
# [span_2](start_span)[span_2](end_span)

# Print the output data (headers are included automatically)

df.show(truncate=False)

df.select("manifest_path","manifest_length","sequence_number","added_snapshot_id","added_files_count","deleted_files_count","added_rows_count",
          "deleted_rows_count","partitions.contains_null","partitions.lower_bound","partitions.upper_bound").orderBy("sequence_number").show(truncate=False)

# [span_3](start_span)[span_3](end_span)


root
 |-- manifest_path: string (nullable = true)
 |-- manifest_length: long (nullable = true)
 |-- partition_spec_id: integer (nullable = true)
 |-- content: integer (nullable = true)
 |-- sequence_number: long (nullable = true)
 |-- min_sequence_number: long (nullable = true)
 |-- added_snapshot_id: long (nullable = true)
 |-- added_files_count: integer (nullable = true)
 |-- existing_files_count: integer (nullable = true)
 |-- deleted_files_count: integer (nullable = true)
 |-- added_rows_count: long (nullable = true)
 |-- existing_rows_count: long (nullable = true)
 |-- deleted_rows_count: long (nullable = true)
 |-- partitions: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- contains_null: boolean (nullable = true)
 |    |    |-- contains_nan: boolean (nullable = true)
 |    |    |-- lower_bound: binary (nullable = true)
 |    |    |-- upper_bound: binary (nullable = true)
 |-- key_metadata: binary (nullable = true)

+------------------------

In [42]:
# Define the full S3A path to your MinIO file
# MANIFEST FILES
manifest_file_path1 = "s3a://warehouse/local/urbanfleet/trips/metadata/f7bf3b16-2d4d-4dd7-bdbd-c4a30c443329-m0.avro"
manifest_file_path2 = "s3a://warehouse/local/urbanfleet/trips/metadata/d2be8abb-e79e-488b-a9a8-57a848a3d80f-m0.avro"
manifest_file_path3 = "s3a://warehouse/local/urbanfleet/trips/metadata/afb2b273-41da-4581-8227-9c8ace235b81-m1.avro"
manifest_file_path4 = "s3a://warehouse/local/urbanfleet/trips/metadata/afb2b273-41da-4581-8227-9c8ace235b81-m0.avro"
# Read the Avro file into a DataFrame
df1 = spark.read.format("avro").load(manifest_file_path1)
df2 = spark.read.format("avro").load(manifest_file_path2)
df3 = spark.read.format("avro").load(manifest_file_path3)
df4 = spark.read.format("avro").load(manifest_file_path4)
# [span_1](start_span)[span_1](end_span)

# Print the schema (this displays all column names and data types)
# df1.printSchema()
# [span_2](start_span)[span_2](end_span)
cols = ["status","snapshot_id", "file_sequence_number", "data_file.file_path", "data_file.partition",
        "data_file.record_count", "data_file.file_size_in_bytes"]
# Print the output data (headers are included automatically)
df1.select(cols).orderBy("sequence_number").show(truncate=False)
df2.select(cols).orderBy("sequence_number").show(truncate=False)
df3.select(cols).orderBy("sequence_number").show(truncate=False)
df4.select(cols).orderBy("sequence_number").show(truncate=False)
# [span_3](start_span)[span_3](end_span)

# status : 1 -> Added , 0 -> Existing, 2 -> deleted.


+------+-------------------+--------------------+--------------------------------------------------------------------------------------------------------------------------------------+------------+------------+------------------+
|status|snapshot_id        |file_sequence_number|file_path                                                                                                                             |partition   |record_count|file_size_in_bytes|
+------+-------------------+--------------------+--------------------------------------------------------------------------------------------------------------------------------------+------------+------------+------------------+
|1     |5493347516158483480|NULL                |s3a://warehouse/local/urbanfleet/trips/data/trip_timestamp_day=2025-01-12/00000-5-d4546b73-8d81-4065-8fc9-ec23f11a42a9-0-00003.parquet|{2025-01-12}|1           |2771              |
|1     |5493347516158483480|NULL                |s3a://warehouse/local/urbanflee

In [43]:
CREATE NAMESPACE IF NOT EXISTS local.urbanfleet;
USE local.urbanfleet;

CREATE TABLE local.urbanfleet.trips (
    trip_id           BIGINT,
    driver_id         BIGINT,
    rider_id          BIGINT,
    pickup_zip        STRING,
    dropoff_zip       STRING,
    fare_amount       DECIMAL(10, 2),
    trip_status       STRING, -- ONGOING, COMPLETED, CANCELLED
    trip_timestamp    TIMESTAMP,
    last_updated      TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(trip_timestamp))
TBLPROPERTIES (
    'format-version' = '2',
    'write.update.mode' = 'merge-on-read',
    'write.delete.mode' = 'merge-on-read',
    'write.merge.mode'  = 'merge-on-read'
);

SyntaxError: invalid syntax (3953095197.py, line 1)

In [44]:
parquet_data_file1="s3a://warehouse/local/urbanfleet/trips/data/trip_timestamp_day=2025-01-12/00000-5-d4546b73-8d81-4065-8fc9-ec23f11a42a9-0-00003.parquet"
parquet_data_file2="s3a://warehouse/local/urbanfleet/trips/data/trip_timestamp_day=2025-01-10/00000-5-d4546b73-8d81-4065-8fc9-ec23f11a42a9-0-00001.parquet"
parquet_data_file3="s3a://warehouse/local/urbanfleet/trips/data/trip_timestamp_day=2025-01-11/00000-5-d4546b73-8d81-4065-8fc9-ec23f11a42a9-0-00002.parquet"


parquet_data_file4="s3a://warehouse/local/urbanfleet/trips/data/trip_timestamp_day=2025-01-11/00000-7-18da0a2e-0769-4b51-8224-33c604a0ad73-00001.parquet"

parquet_data_file5="s3a://warehouse/local/urbanfleet/trips/data/trip_timestamp_day=2025-01-11/00000-11-bced53f4-866e-4cec-b2ea-800bdd0867a1-00001-deletes.parquet"
parquet_data_file6="s3a://warehouse/local/urbanfleet/trips/data/trip_timestamp_day=2025-01-11/00000-7-18da0a2e-0769-4b51-8224-33c604a0ad73-00001-deletes.parquet"

data_df1= spark.read.parquet(parquet_data_file1)
data_df2= spark.read.parquet(parquet_data_file2)
data_df3= spark.read.parquet(parquet_data_file3)
data_df4= spark.read.parquet(parquet_data_file4)
data_df5= spark.read.parquet(parquet_data_file5)
data_df6= spark.read.parquet(parquet_data_file6)

delete_files1= "s3a://warehouse/local/urbanfleet/trips/data/trip_timestamp_day=2025-01-11/00000-5-d4546b73-8d81-4065-8fc9-ec23f11a42a9-0-00002.parquet"
del_data_df1= spark.read.parquet(delete_files1)

data_df1.show(truncate=False)
data_df2.show(truncate=False)
data_df3.show(truncate=False)
data_df4.show(truncate=False)
data_df5.show(truncate=False)
data_df6.show(truncate=False)

print("***************")
del_data_df1.show(truncate=False)


# |s3a://warehouse/local/urbanfleet/trips/data/trip_timestamp_day=2025-01-11/00000-5-d4546b73-8d81-4065-8fc9-ec23f11a42a9-0-00002.parquet|0  |
# |s3a://warehouse/local/urbanfleet/trips/data/trip_timestamp_day=2025-01-11/00000-5-d4546b73-8d81-4065-8fc9-ec23f11a42a9-0-00002.parquet|1  |
# +--------------------------------------------------------------------------------------------------------------------------------------+---+

# +--------------------------------------------------------------------------------------------------------------------------------------+---+
# |file_path                                                                                                                             |pos|
# +--------------------------------------------------------------------------------------------------------------------------------------+---+
# |s3a://warehouse/local/urbanfleet/trips/data/trip_timestamp_day=2025-01-11/00000-5-d4546b73-8d81-4065-8fc9-ec23f11a42a9-0-00002.parquet|0  |
# +--------------------------------------------------------------------------------------------------------------------------------------+---+


+-------+---------+--------+----------+-----------+-----------+-----------+-------------------+-------------------+
|trip_id|driver_id|rider_id|pickup_zip|dropoff_zip|fare_amount|trip_status|trip_timestamp     |last_updated       |
+-------+---------+--------+----------+-----------+-----------+-----------+-------------------+-------------------+
|105    |5004     |9005    |10008     |10009      |45.00      |COMPLETED  |2025-01-12 22:00:00|2025-01-12 22:45:00|
+-------+---------+--------+----------+-----------+-----------+-----------+-------------------+-------------------+

+-------+---------+--------+----------+-----------+-----------+-----------+-------------------+-------------------+
|trip_id|driver_id|rider_id|pickup_zip|dropoff_zip|fare_amount|trip_status|trip_timestamp     |last_updated       |
+-------+---------+--------+----------+-----------+-----------+-----------+-------------------+-------------------+
|101    |5001     |9001    |10001     |10002      |25.50      |COMPLETE

Here is a detailed breakdown of the `metadata.json` snippet you shared, explained in the context of your specific `trips` table scenario, followed by a clarification of the manifest files.

### 1. Breakdown of the Snapshot JSON Parameters

This JSON represents **Snapshot 3**, which was created when you ran a `DELETE` (or `UPDATE`) SQL query on your `trips` table.

* **`sequence-number: 3`**: This is the 3rd modification made to the table. (1 was the initial insert, 2 was likely an update/insert, and 3 is this current delete/update operation).
* **`snapshot-id: 3529268444000415922`**: The unique identifier for this specific point-in-time state of your table.
* **`parent-snapshot-id: 9030272551482834429`**: This points back to Snapshot 2. It tells Iceberg, "To understand the full table, start with Snapshot 2, and then apply the changes I am about to describe."
* **`summary.operation: "delete"`**: The Spark SQL operation that triggered this snapshot resulted in deleting rows (in a Merge-On-Read table, updates are often implemented as a delete + an insert).
* **`summary.added-position-delete-files: "1"` & `added-delete-files: "1"**`: Your operation created **1 new** Parquet delete file.
* **`summary.removed-position-delete-files: "1"` & `removed-delete-files: "1"**`: Iceberg actively marked **1 older** Parquet delete file as obsolete/removed. *(Note: Look closely at the JSON you pasted above—these exact lines are indeed in your snippet!)*
* **`summary.added-position-deletes: "2"`**: The newly created delete file contains the physical locations of **2 specific rows** that should be ignored when reading the data.
* **`summary.removed-position-deletes: "1"`**: The old delete file that was removed contained only **1 row** deletion.
* **`summary.total-data-files: "4"` & `total-records: "6"**`: Because this is a Merge-On-Read table, Iceberg did not delete or modify your actual base data files (`.parquet`). You still have 4 data files containing 6 total historical rows on disk. The reader will just use the delete files to filter out the 2 deleted rows on the fly.
* **`summary.total-delete-files: "1"`**: Since Iceberg added 1 delete file and removed 1 delete file in this snapshot, the net total of *active* delete files for the entire table remains 1.

---

### 2. Clarifying the Two Manifest Files (`-m1.avro` and `-m0.avro`)

To understand why Iceberg generated two separate manifest files for Snapshot 3, look at the dataframe output from **Cell `[42]**` in your notebook screenshot.

When you ran your SQL query, Iceberg realized it needed to track 2 deleted rows for the `2025-01-11` partition. Instead of keeping the old delete file (which only had 1 row) and writing a second small delete file, Iceberg combined them. It wrote a brand new delete file containing *both* deleted rows.

Because manifest files are strictly **immutable** (they cannot be edited once written), Iceberg had to write two new manifest files to record this swap:

1. **The `-m1.avro` Manifest (The Addition):**
* If you look at the 3rd dataframe in Cell `[42]`, it shows `status=1` (ADDED).
* This manifest introduces the newly created delete file (`...-00001-deletes.parquet`, size `1687` bytes).
* This file contains the 2 deleted rows (`added-position-deletes: 2`).


2. **The `-m0.avro` Manifest (The Tombstone/Deletion):**
* If you look at the 4th dataframe in Cell `[42]`, it shows `status=2` (DELETED).
* This manifest acts as a tombstone. It tells Iceberg to stop using the old delete file from Snapshot 2 (`size: 1636` bytes, which contained 1 deleted row).
* Even though the physical file might still exist on MinIO until you run a `VACUUM` or expire snapshots, this manifest tells the Iceberg reader to completely ignore it.



So, the `manifest-list` for Snapshot 3 points to both of these files so the reader knows exactly which new file to use and which old file to ignore.

In [31]:
parquet_data_file="s3a://warehouse/local/urbanfleet/trips/metadata/d2be8abb-e79e-488b-a9a8-57a848a3d80f-m0.avro"

data_df= spark.read.format(parquet_data_file)

data_df.show(truncate=False)

26/10/10 13:58:23 ERROR Executor: Exception in task 0.0 in stage 28.0 (TID 28)
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.util.ThreadUtils$.parmap(ThreadUtils.scala:387)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.readParquetFootersInParallel(ParquetFileFormat.scala:443)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1(ParquetFileFormat.scala:493)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1$adapted(ParquetFileFormat.scala:485)
	at org.apache.spark.sql.execution.datasources.SchemaMergeUtils$.$anonfun$mergeSchemasInParallel$2(SchemaMergeUtils.scala:80)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitions$2(RDD.scala:858)
	at org.apache.spar

Py4JJavaError: An error occurred while calling o289.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 28.0 failed 1 times, most recent failure: Lost task 0.0 in stage 28.0 (TID 28) (772b89eb2a63 executor driver): org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.util.ThreadUtils$.parmap(ThreadUtils.scala:387)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.readParquetFootersInParallel(ParquetFileFormat.scala:443)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1(ParquetFileFormat.scala:493)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1$adapted(ParquetFileFormat.scala:485)
	at org.apache.spark.sql.execution.d

In [19]:
manifest_file_1="s3a://warehouse/local/urbanfleet/trips/metadata/afb2b273-41da-4581-8227-9c8ace235b81-m0.avro"
df = spark.read.format("avro").load(manifest_file_1)

df.printSchema()
# [span_2](start_span)[span_2](end_span)

# Print the output data (headers are included automatically)
df.orderBy("sequence_number").show(truncate=False)



root
 |-- status: integer (nullable = true)
 |-- snapshot_id: long (nullable = true)
 |-- sequence_number: long (nullable = true)
 |-- file_sequence_number: long (nullable = true)
 |-- data_file: struct (nullable = true)
 |    |-- content: integer (nullable = true)
 |    |-- file_path: string (nullable = true)
 |    |-- file_format: string (nullable = true)
 |    |-- partition: struct (nullable = true)
 |    |    |-- trip_timestamp_day: date (nullable = true)
 |    |-- record_count: long (nullable = true)
 |    |-- file_size_in_bytes: long (nullable = true)
 |    |-- column_sizes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- key: integer (nullable = true)
 |    |    |    |-- value: long (nullable = true)
 |    |-- value_counts: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- key: integer (nullable = true)
 |    |    |    |-- value: long (nullable = true)
 |    |-- null_value_counts: arr

In [ ]:
delete_file1= "s3a://warehouse/local/urbanfleet/trips/data/trip_timestamp_day=2025-01-11/00000-7-18da0a2e-0769-4b51-8224-33c604a0ad73-00001-deletes.parquet"

df=spark.read.parquet(delete_file1)

df.show(truncate=False)

delete_file2= "s3a://warehouse/local/urbanfleet/trips/data/trip_timestamp_day=2025-01-11/00000-11-bced53f4-866e-4cec-b2ea-800bdd0867a1-00001-deletes.parquet"
df=spark.read.parquet(delete_file2)

df.show(truncate=False)


In [2]:
# 2. Create a Namespace (Schema)
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.training")
print("Namespace 'demo.training' ready.")

# 3. Create an Iceberg Table
spark.sql("""
    CREATE TABLE IF NOT EXISTS demo.training.employee (
        emp_id BIGINT,
        emp_name STRING,
        department STRING,
        salary DOUBLE
    ) USING iceberg
""")
print("Iceberg table 'employee' ready.")

26/09/13 16:25:58 WARN JdbcCatalog: JDBC catalog is initialized without view support. To auto-migrate the database's schema and enable view support, set jdbc.schema-version=V1


Namespace 'demo.training' ready.
Iceberg table 'employee' ready.


In [3]:
spark.sql("Select * from demo.training.employee")

DataFrame[emp_id: bigint, emp_name: string, department: string, salary: double]

In [4]:
# 4. Write Data into the Table
spark.sql("""
    INSERT INTO demo.training.employee 
    VALUES 
        (1, 'Ashish Singh', 'Data Engineering', 150000.00),
        (2, 'Preeti Singh', 'Analytics', 135000.00)
""")
print("Data successfully committed to MinIO.")

# 5. Query the Active Data
print("\n--- CURRENT TABLE DATA ---")
spark.sql("SELECT * FROM demo.training.employee").show()

Data successfully committed to MinIO.

--- CURRENT TABLE DATA ---
+------+------------+----------------+--------+
|emp_id|    emp_name|      department|  salary|
+------+------------+----------------+--------+
|     1|Ashish Singh|Data Engineering|150000.0|
|     2|Preeti Singh|       Analytics|135000.0|
+------+------------+----------------+--------+



In [5]:

# ==========================================
# 6. VISUALIZE ICEBERG METADATA & HISTORY
# ==========================================

print("\n--- TABLE HISTORY (Timeline of Commits) ---")
# Shows when the table was created and every subsequent commit timestamp
spark.sql("SELECT * FROM demo.training.employee.history").show(truncate=False)

print("\n--- TABLE SNAPSHOTS ---")
# Shows the specific operations (e.g., 'append', 'overwrite') and row counts for each commit
spark.sql("SELECT snapshot_id, operation, summary FROM demo.training.employee.snapshots").show(truncate=False)

print("\n--- PHYSICAL DATA FILES ---")
# Exposes the actual Parquet files sitting in your MinIO 'warehouse' bucket
spark.sql("SELECT file_path, file_size_in_bytes, record_count FROM demo.training.employee.files").show(truncate=False)


--- TABLE HISTORY (Timeline of Commits) ---
+-----------------------+-------------------+---------+-------------------+
|made_current_at        |snapshot_id        |parent_id|is_current_ancestor|
+-----------------------+-------------------+---------+-------------------+
|2026-09-13 16:26:28.906|7001475892194027695|NULL     |true               |
+-----------------------+-------------------+---------+-------------------+


--- TABLE SNAPSHOTS ---
+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |operation|summary                     

In [6]:
# Create a DataFrame
data = [(3, 'Ravi Kumar', 'Cloud Infrastructure', 1400200.00)]
columns = ["emp_id", "emp_name", "department", "salary"]
df = spark.createDataFrame(data, columns)

# Write to Iceberg using the V2 API
df.writeTo("demo.training.employee").append()

# Run the snapshots query again to see the new 'append' operation logged
spark.sql("SELECT snapshot_id, operation, summary FROM demo.training.employee.snapshots").show(truncate=False)

+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |operation|summary                                                                                                                                                                                                                                                                                                                                                                                                                                                                  |
+-------------------

In [6]:
spark.sql("SELECT * FROM demo.training.employee.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-09-03 17:02:53.083|2144457049189745365|NULL               |true               |
|2026-09-03 17:03:01.438|3568146871021356834|2144457049189745365|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [7]:
%%sql
    
SELECT * FROM demo.training.employee.history

26/09/03 17:03:06 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


made_current_at,snapshot_id,parent_id,is_current_ancestor
2026-09-03 17:02:53.083000,2144457049189745365,None,True
2026-09-03 17:03:01.438000,3568146871021356834,2144457049189745365,True


In [8]:
%%sql
SELECT * FROM demo.training.employee version as of 2144457049189745365;


emp_id,emp_name,department,salary
1,Ashish Singh,Data Engineering,150000.0
2,Preeti Singh,Analytics,135000.0
